In [1]:
import pandas as pd
import requests
import time
import os
from tqdm import tqdm

OVERPASS_SERVERS = [
    "https://overpass-api.de/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter"
]

headers = {
    "User-Agent": "RentoraAI/1.0 (https://github.com/anisha-1811/Rentora-AI)"
}

CACHE_FILE = "../data/geo_features_cache.csv"
FAILED_FILE = "../data/geo_features_failed.csv"

In [2]:
combined = pd.read_csv("../data/rentora_model_ready.csv", low_memory=False)
print(combined.shape)

unique_locs = combined[["city", "locality", "latitude", "longitude"]].drop_duplicates(subset=["latitude", "longitude"])
print("Unique locations:", len(unique_locs))

if os.path.exists(CACHE_FILE):
    cache = pd.read_csv(CACHE_FILE)
    print("Loaded cache:", len(cache))
else:
    cache = pd.DataFrame(columns=["latitude", "longitude", "near_highway", "near_mall", "near_river", "near_mountain"])

processed = set(zip(cache.latitude, cache.longitude))
remaining = unique_locs[~unique_locs.set_index(["latitude", "longitude"]).index.isin(processed)]
print("Remaining:", len(remaining))

(117443, 10)
Unique locations: 7765
Remaining: 7765


In [3]:
def get_geo_features(lat, lon, radius=2000):
    if pd.isna(lat) or pd.isna(lon):
        return {"near_highway": None, "near_mall": None, "near_river": None, "near_mountain": None}

    query = f"""
[out:json][timeout:25];
(
way["highway"~"trunk|primary|motorway"](around:{radius},{lat},{lon});
node["shop"="mall"](around:{radius},{lat},{lon});
way["natural"="water"](around:{radius},{lat},{lon});
node["natural"="peak"](around:{radius},{lat},{lon});
);
out tags;
"""

    for attempt in range(3):
        for url in OVERPASS_SERVERS:
            try:
                response = requests.post(url, data=query, headers=headers, timeout=120)

                if response.status_code in [429, 500, 502, 503, 504]:
                    wait = (attempt + 1) * 10
                    print(f"{response.status_code} from {url}. Waiting {wait}s...")
                    time.sleep(wait)
                    continue

                response.raise_for_status()
                data = response.json()
                elements = data.get("elements", [])

                return {
                    "near_highway": any(e.get("tags", {}).get("highway") in ["motorway", "primary", "trunk"] for e in elements),
                    "near_mall": any(e.get("tags", {}).get("shop") == "mall" for e in elements),
                    "near_river": any(e.get("tags", {}).get("natural") == "water" for e in elements),
                    "near_mountain": any(e.get("tags", {}).get("natural") == "peak" for e in elements)
                }

            except Exception as e:
                print(f"Error at {url}: {e}")
                continue

    return {"near_highway": None, "near_mall": None, "near_river": None, "near_mountain": None}

In [4]:
headers = {
    "User-Agent": "RentoraAI/1.0 (https://github.com/anisha-1811/Rentora-AI)"
}

In [5]:
def get_geo_features(lat, lon, radius=2000):

    query = f"""
[out:json][timeout:25];
(
way["highway"~"trunk|primary|motorway"](around:{radius},{lat},{lon});
node["shop"="mall"](around:{radius},{lat},{lon});
way["natural"="water"](around:{radius},{lat},{lon});
node["natural"="peak"](around:{radius},{lat},{lon});
);
out tags;
"""

    for attempt in range(5):

        try:

            response = requests.post(
                "https://overpass-api.de/api/interpreter",
                data=query,
                headers=headers,
                timeout=60
            )

            if response.status_code == 429:

                wait = (attempt + 1) * 10

                print(f"Rate limited. Waiting {wait} sec...")

                time.sleep(wait)

                continue

            response.raise_for_status()

            data = response.json()

            elements = data.get("elements", [])

            return {
                "near_highway": any(
                    e.get("tags", {}).get("highway")
                    in ["motorway", "primary", "trunk"]
                    for e in elements
                ),

                "near_mall": any(
                    e.get("tags", {}).get("shop") == "mall"
                    for e in elements
                ),

                "near_river": any(
                    e.get("tags", {}).get("natural") == "water"
                    for e in elements
                ),

                "near_mountain": any(
                    e.get("tags", {}).get("natural") == "peak"
                    for e in elements
                )

            }

        except Exception as e:

            print(e)

            time.sleep(10)

    return {
        "near_highway": None,
        "near_mall": None,
        "near_river": None,
        "near_mountain": None
    }

In [6]:
processed = set(zip(cache.latitude, cache.longitude))

remaining = unique_locs[
    ~unique_locs.set_index(["latitude","longitude"]).index.isin(processed)
]

print("Remaining:", len(remaining))

Remaining: 7765


In [8]:
# Process only the first 10 remaining locations
test_remaining = remaining.head(5)

new_rows = []

count = 0

for _, row in tqdm(test_remaining.iterrows(), total=len(test_remaining)):

    print(f"\nProcessing {count + 1}/5 : {row['city']} - {row['locality']}")

    features = get_geo_features(
        row.latitude,
        row.longitude
    )

    new_rows.append({
        "latitude": row.latitude,
        "longitude": row.longitude,
        **features
    })

    count += 1

    print(features)

    # Wait 2 seconds to avoid rate limiting
    time.sleep(2)

# Convert results to DataFrame
temp = pd.DataFrame(new_rows)

print("\nFinished!")
print(temp)

  0%|          | 0/5 [00:00<?, ?it/s]


Processing 1/5 : Ahmedabad - Bodakdev
504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
{'near_highway': True, 'near_mall': True, 'near_river': True, 'near_mountain': False}


 20%|██        | 1/5 [00:32<02:09, 32.44s/it]


Processing 2/5 : Ahmedabad - CG Road
504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
{'near_highway': True, 'near_mall': False, 'near_river': True, 'near_mountain': False}


 40%|████      | 2/5 [01:31<02:24, 48.12s/it]

...


Processing 4/5 : Ahmedabad - Sanand
Rate limited. Waiting 10 sec...
{'near_highway': True, 'near_mall': False, 'near_river': True, 'near_mountain': False}


 80%|████████  | 4/5 [02:53<00:43, 43.21s/it]


Processing 5/5 : Ahmedabad - Navrangpura
Rate limited. Waiting 10 sec...
{'near_highway': True, 'near_mall': False, 'near_river': True, 'near_mountain': False}


100%|██████████| 5/5 [03:30<00:00, 42.15s/it]


Finished!
    latitude  longitude  near_highway  near_mall  near_river  near_mountain
0  23.044592  72.517344          True       True        True          False
1  23.026011  72.556718          True      False        True          False
2  23.016927  72.520432          True       True        True          False
3  23.023888  72.385148          True      False        True          False
4  23.036000  72.564343          True      False        True          False
